# 🌿 04 Evaluasi — Sistem LSTM Bonsai

| Item       | Detail                                      |
|------------|---------------------------------------------|
| **File**   | `notebooks/04_evaluasi.ipynb`                |
| **Tujuan** | Hitung semua metrik evaluasi, buat visualisasi, simpan laporan. |
| **Input**  | `artifacts/predictions.csv`, `artifacts/y_test_labels.csv`, `artifacts/training_history.csv` |
| **Output** | `output/01_confusion_matrix.png`, `output/02_roc_curve.png`, `output/03_training_history.png`, `output/04_prediction_vs_actual.png`, `output/05_residual_plot.png`, `output/06_classification_report.csv`, `output/07_metrics_summary.csv` |
| **Urutan** | Jalankan setelah: `03_testing.ipynb` |

In [1]:
# ── VERIFIKASI LINGKUNGAN ──────────────────────────────────────────────
import sys, os

assert ".venv" in sys.executable or "venv" in sys.executable, (
    "⛔ Kernel bukan dari .venv!\n"
    "Ganti kernel ke: Python (bonsai-lstm)\n"
    f"Kernel saat ini: {sys.executable}"
)
print("✅ Kernel  :", sys.executable)
print("✅ Python  :", sys.version)
# ──────────────────────────────────────────────────────────────────────

✅ Kernel  : d:\bonsai-lstm\.venv\Scripts\python.exe
✅ Python  : 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]


In [2]:
# ── IMPORT LIBRARY, KONSTANTA, SEED ───────────────────────────────────
import random, os
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, auc, accuracy_score,
    mean_squared_error, mean_absolute_error, f1_score
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
print(f"[SEED] Global random seed = {SEED}")

ARTIFACTS_DIR = "../artifacts"
OUTPUT_DIR    = "../output"
DPI           = 150
N_SHOW        = 200

TARGET = {
    "accuracy" : 0.85,
    "f1"       : 0.80,
    "auc_roc"  : 0.85,
    "rmse_pct" : 5.0,
    "mae_pct"  : 3.0,
}

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("✅ Konfigurasi evaluasi siap.")

[SEED] Global random seed = 42
✅ Konfigurasi evaluasi siap.


In [3]:
# ── RULE-NB-06: Validasi Artefak ───────────────────────────────────────
REQUIRED = [
    f"{ARTIFACTS_DIR}/predictions.csv",
    f"{ARTIFACTS_DIR}/y_test_labels.csv",
    f"{ARTIFACTS_DIR}/training_history.csv",
]
missing = [f for f in REQUIRED if not os.path.exists(f)]
assert not missing, f"⛔ Artefak tidak ada: {missing}"
print("✅ Semua artefak tersedia.")

✅ Semua artefak tersedia.


In [4]:
# ── RULE-EVAL-02: Muat Data Evaluasi ───────────────────────────────────
pred_df   = pd.read_csv(f"{ARTIFACTS_DIR}/predictions.csv")
actual_df = pd.read_csv(f"{ARTIFACTS_DIR}/y_test_labels.csv")
hist_df   = pd.read_csv(f"{ARTIFACTS_DIR}/training_history.csv")

y_true       = actual_df["y_true_class"].values
y_true_soil  = actual_df["y_true_soil_pct"].values
y_pred_prob  = pred_df["y_pred_prob"].values
y_pred_class = pred_df["y_pred_class"].values
y_pred_soil  = pred_df["y_pred_soil_pct"].values
print("✅ Data evaluasi dimuat.")

✅ Data evaluasi dimuat.


In [5]:
# ── RULE-EVAL-03: Hitung Metrik Regresi ────────────────────────────────
rmse = np.sqrt(mean_squared_error(np.asarray(y_true_soil, dtype=float), np.asarray(y_pred_soil, dtype=float)))
mae  = mean_absolute_error(np.asarray(y_true_soil, dtype=float), np.asarray(y_pred_soil, dtype=float))

print(f"[RMSE] {rmse:.4f}% | Target ≤ {TARGET['rmse_pct']}% → {'✅ PASS' if rmse<=TARGET['rmse_pct'] else '❌ FAIL'}")
print(f"[MAE ] {mae:.4f}%  | Target ≤ {TARGET['mae_pct']}%  → {'✅ PASS' if mae<=TARGET['mae_pct']  else '❌ FAIL'}")

[RMSE] 69.9386% | Target ≤ 5.0% → ❌ FAIL
[MAE ] 65.8414%  | Target ≤ 3.0%  → ❌ FAIL


In [6]:
# ── RULE-EVAL-04: Hitung Metrik Klasifikasi ────────────────────────────
acc     = accuracy_score(np.asarray(y_true), np.asarray(y_pred_class))
f1      = f1_score(np.asarray(y_true), np.asarray(y_pred_class), average="weighted")
fpr, tpr, _ = roc_curve(np.asarray(y_true), np.asarray(y_pred_prob))
roc_auc = auc(fpr, tpr)

print(f"[ACC    ] {acc:.4f} | Target ≥ {TARGET['accuracy']}  → {'✅ PASS' if acc>=TARGET['accuracy'] else '❌ FAIL'}")
print(f"[F1     ] {f1:.4f} | Target ≥ {TARGET['f1']}       → {'✅ PASS' if f1>=TARGET['f1'] else '❌ FAIL'}")
print(f"[AUC-ROC] {roc_auc:.4f} | Target ≥ {TARGET['auc_roc']} → {'✅ PASS' if roc_auc>=TARGET['auc_roc'] else '❌ FAIL'}")

[ACC    ] 0.9131 | Target ≥ 0.85  → ✅ PASS
[F1     ] 0.9148 | Target ≥ 0.8       → ✅ PASS
[AUC-ROC] 0.9864 | Target ≥ 0.85 → ✅ PASS


In [7]:
# ── RULE-EVAL-05: Confusion Matrix ──────────────────────────────────────
def plot_confusion_matrix(y_true, y_pred, path, dpi):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    prec = tp/(tp+fp) if (tp+fp) else 0
    rec  = tp/(tp+fn) if (tp+fn) else 0
    f1_  = 2*prec*rec/(prec+rec) if (prec+rec) else 0

    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Tidak Siram","Siram"],
                yticklabels=["Tidak Siram","Siram"],
                linewidths=1, linecolor="gray", ax=ax, annot_kws={"size":14})
    ax.set_xlabel("Prediksi", fontsize=12)
    ax.set_ylabel("Aktual",   fontsize=12)
    ax.set_title("Confusion Matrix — LSTM Bonsai", fontsize=14, fontweight="bold")
    info = (f"TP={tp}  FP={fp}  FN={fn}  TN={tn}\n"
            f"Precision={prec:.4f}  Recall={rec:.4f}  F1-Score={f1_:.4f}")
    ax.text(0.5, -0.18, info, transform=ax.transAxes, ha="center", fontsize=10,
            bbox=dict(boxstyle="round,pad=0.4", facecolor="lightyellow", edgecolor="gray"))
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close()
    print(f"[VIZ] ✅ {path}")

plot_confusion_matrix(y_true, y_pred_class, f"{OUTPUT_DIR}/01_confusion_matrix.png", DPI)

[VIZ] ✅ ../output/01_confusion_matrix.png


In [8]:
# ── RULE-EVAL-06: ROC Curve ─────────────────────────────────────────────
def plot_roc(fpr, tpr, roc_auc, path, dpi):
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.plot(fpr, tpr, color="steelblue", lw=2, label=f"ROC (AUC = {roc_auc:.4f})")
    ax.plot([0,1],[0,1], color="gray", linestyle="--", lw=1.5, label="Random Classifier")
    ax.fill_between(fpr, tpr, alpha=0.15, color="steelblue")
    ax.set_xlabel("False Positive Rate", fontsize=12)
    ax.set_ylabel("True Positive Rate",  fontsize=12)
    ax.set_title("ROC Curve — LSTM Bonsai", fontsize=14, fontweight="bold")
    ax.legend(loc="lower right", fontsize=11); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close()
    print(f"[VIZ] ✅ {path}")

plot_roc(fpr, tpr, roc_auc, f"{OUTPUT_DIR}/02_roc_curve.png", DPI)

[VIZ] ✅ ../output/02_roc_curve.png


In [9]:
# ── RULE-EVAL-07: Training History ───────────────────────────────────────
def plot_history(hist_df, path, dpi):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle("Training History — LSTM Bonsai", fontsize=14, fontweight="bold")

    axes[0].plot(hist_df["loss"],         label="Train Loss",    color="tomato")
    axes[0].plot(hist_df["val_loss"],     label="Val Loss",      color="steelblue", ls="--")
    axes[0].set_title("Loss per Epoch");  axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
    axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(hist_df["accuracy"],     label="Train Accuracy",color="seagreen")
    axes[1].plot(hist_df["val_accuracy"], label="Val Accuracy",  color="orange",    ls="--")
    axes[1].set_title("Accuracy per Epoch"); axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
    axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close()
    print(f"[VIZ] ✅ {path}")

plot_history(hist_df, f"{OUTPUT_DIR}/03_training_history.png", DPI)

[VIZ] ✅ ../output/03_training_history.png


In [10]:
# ── RULE-EVAL-08: Prediksi vs Aktual & Residual Plot ────────────────────
def plot_pred_vs_actual(y_act, y_pred, n, path, dpi):
    fig, ax = plt.subplots(figsize=(13, 5))
    ax.plot(y_act[:n],  label="Aktual",   color="steelblue", lw=1.5)
    ax.plot(y_pred[:n], label="Prediksi", color="tomato",    lw=1.5, ls="--")
    ax.axhline(60, color="green", ls=":", lw=1.5, label="Threshold 60%")
    ax.set_xlabel("Timestep"); ax.set_ylabel("Kelembaban Tanah (%)")
    ax.set_title(f"Prediksi vs Aktual ({n} sampel) — LSTM Bonsai", fontsize=14, fontweight="bold")
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close()
    print(f"[VIZ] ✅ {path}")

def plot_residual(y_act, y_pred, path, dpi):
    residuals = y_act - y_pred
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle("Residual Plot — Error Prediksi LSTM Bonsai", fontsize=14, fontweight="bold")

    axes[0].plot(residuals, color="purple", lw=0.8, alpha=0.7)
    axes[0].axhline(0, color="black", ls="--", lw=1)
    axes[0].fill_between(range(len(residuals)), residuals, alpha=0.15, color="purple")
    axes[0].set_xlabel("Timestep"); axes[0].set_ylabel("Aktual - Prediksi")
    axes[0].set_title("Residual per Timestep"); axes[0].grid(alpha=0.3)

    axes[1].hist(residuals, bins=40, color="purple", alpha=0.7, edgecolor="white")
    axes[1].axvline(0, color="black", ls="--", lw=1.5)
    axes[1].set_xlabel("Error"); axes[1].set_ylabel("Frekuensi")
    axes[1].set_title("Distribusi Error (Histogram)"); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close()
    print(f"[VIZ] ✅ {path}")

plot_pred_vs_actual(y_true_soil, y_pred_soil, N_SHOW, f"{OUTPUT_DIR}/04_prediction_vs_actual.png", DPI)
plot_residual(y_true_soil, y_pred_soil, f"{OUTPUT_DIR}/05_residual_plot.png", DPI)

[VIZ] ✅ ../output/04_prediction_vs_actual.png
[VIZ] ✅ ../output/05_residual_plot.png


In [11]:
# ── RULE-EVAL-09: Simpan Classification Report & Metrics Summary ───────
report = classification_report(
    np.asarray(y_true), np.asarray(y_pred_class),
    target_names=["Tidak Siram (0)", "Siram (1)"],
    output_dict=True, digits=4
)
pd.DataFrame(report).T.to_csv(f"{OUTPUT_DIR}/06_classification_report.csv")
print(classification_report(np.asarray(y_true), np.asarray(y_pred_class),
      target_names=["Tidak Siram (0)", "Siram (1)"], digits=4))
print(f"[SAVE] ✅ output/06_classification_report.csv")

summary_df = pd.DataFrame({
    "Metric" : ["Accuracy", "F1-Score (weighted)", "AUC-ROC", "RMSE (%)", "MAE (%)"],
    "Value"  : [round(float(acc),4), round(float(f1),4), round(float(roc_auc),4), round(float(rmse),4), round(float(mae),4)],
    "Target" : ["≥ 0.85", "≥ 0.80", "≥ 0.85", "≤ 5.0", "≤ 3.0"],
    "Status" : [
        "PASS" if acc     >= TARGET["accuracy"]  else "FAIL",
        "PASS" if f1      >= TARGET["f1"]         else "FAIL",
        "PASS" if roc_auc >= TARGET["auc_roc"]    else "FAIL",
        "PASS" if rmse    <= TARGET["rmse_pct"]   else "FAIL",
        "PASS" if mae     <= TARGET["mae_pct"]    else "FAIL",
    ]
})
summary_df.to_csv(f"{OUTPUT_DIR}/07_metrics_summary.csv", index=False)
print(summary_df.to_string(index=False))
print(f"[SAVE] ✅ output/07_metrics_summary.csv")

                 precision    recall  f1-score   support

Tidak Siram (0)     0.9753    0.8955    0.9337       574
      Siram (1)     0.8083    0.9511    0.8739       266

       accuracy                         0.9131       840
      macro avg     0.8918    0.9233    0.9038       840
   weighted avg     0.9224    0.9131    0.9148       840

[SAVE] ✅ output/06_classification_report.csv
             Metric   Value Target Status
           Accuracy  0.9131 ≥ 0.85   PASS
F1-Score (weighted)  0.9148 ≥ 0.80   PASS
            AUC-ROC  0.9864 ≥ 0.85   PASS
           RMSE (%) 69.9386  ≤ 5.0   FAIL
            MAE (%) 65.8414  ≤ 3.0   FAIL
[SAVE] ✅ output/07_metrics_summary.csv


## 📊 Ringkasan Evaluasi

**Visualisasi yang dihasilkan:**
- `01_confusion_matrix.png` → Confusion matrix heatmap
- `02_roc_curve.png` → ROC curve dengan AUC
- `03_training_history.png` → Loss & accuracy per epoch
- `04_prediction_vs_actual.png` → Grafik prediksi vs aktual
- `05_residual_plot.png` → Residual & distribusi error

**Laporan yang dihasilkan:**
- `06_classification_report.csv` → Precision, recall, f1-score per kelas
- `07_metrics_summary.csv` → Ringkasan metrik dengan status PASS/FAIL

**Sistem Bonsai LSTM telah selesai!**
Semua notebook telah dijalankan dan output dihasilkan di folder `output/`.